# Node Extractor V4 — Attention-Pooled Patch Tokens → Residual MLP

Multi-label classifier V4: pools **729 spatial patch tokens** via learnable attention
into a single vector, then decodes through a deep residual MLP.

## Model lineage

| | Input | Architecture | Storage/image |
|---|---|---|---|
| V1 | pooled 1152-d | cross-attention (fake 16 tokens) | 2.3 KB |
| V2 | pooled 1152-d | residual MLP | 2.3 KB |
| **V3** | **729 patches × 1152-d** | **cross-attention (real spatial)** | **~1.6 MB** |
| **V4** | **729 patches × 1152-d** | **attention-pool → residual MLP** | **~1.6 MB** |

V4 answers: *does the spatial patch information help even without per-label attention?*
If V4 ≈ V3, the spatial signal is in the pooling; if V3 >> V4, per-label attention matters.

---
## 1. Setup & Configuration

In [ ]:
import sys, os, time, random
from pathlib import Path
from dataclasses import dataclass

import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "poc_scripts"))

@dataclass
class Config:
    # Paths — reuses V3 patch caches and V1 vocabulary
    train_jsonl : str = str(PROJECT_ROOT / "data" / "merged" / "train.jsonl")
    val_jsonl   : str = str(PROJECT_ROOT / "data" / "merged" / "val.jsonl")
    train_cache : str = str(PROJECT_ROOT / "data" / "merged" / "train_embeddings_patched.h5")
    val_cache   : str = str(PROJECT_ROOT / "data" / "merged" / "val_embeddings_patched.h5")
    vocab_path  : str = str(PROJECT_ROOT / "data" / "merged" / "node_vocab.json")
    checkpoint_dir  : str = str(PROJECT_ROOT / "data" / "merged" / "checkpoints")
    checkpoint_name : str = "best_model_v4.pt"

    # Vocabulary
    min_label_freq : int = 50

    # Model V4 — Attention-Pool + Residual MLP
    patch_dim     : int   = 1152
    hidden_dim    : int   = 2048
    num_blocks    : int   = 6
    expansion     : int   = 2
    dropout       : float = 0.1
    input_dropout : float = 0.05
    pool_mode     : str   = "attention"  # "attention" or "mean"

    # Loss (same ASL as all versions)
    gamma_neg       : float = 4.0
    gamma_pos       : float = 1.0
    clip            : float = 0.05
    label_smoothing : float = 0.05

    # Training — MLP activations are tiny, so large batch is safe
    # Input loading is (B, 729, 1152) but it gets pooled immediately to (B, 1152)
    batch_size       : int   = 128
    eval_batch_size  : int   = 256
    lr               : float = 1e-3
    weight_decay     : float = 0.01
    epochs           : int   = 500
    warmup_ratio     : float = 0.05
    grad_clip        : float = 1.0
    patience         : int   = 10
    num_workers      : int   = 8
    eval_threshold   : float = 0.5
    use_amp          : bool  = True
    compile_model    : bool  = False

cfg = Config()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU:        {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {p.total_mem / 1024**3:.1f} GB")
    print(f"VRAM free:  {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

os.makedirs(cfg.checkpoint_dir, exist_ok=True)
best_path = os.path.join(cfg.checkpoint_dir, cfg.checkpoint_name)
print(f"\nCheckpoint : {best_path}")
print(f"Batch size : {cfg.batch_size}  |  hidden_dim: {cfg.hidden_dim}  |  "
      f"blocks: {cfg.num_blocks}  |  pool: {cfg.pool_mode}")

---
## 2. Load Vocabulary & Datasets

Reuses V3's patch-token HDF5 cache (`train_embeddings_patched.h5`) and V1's vocabulary.
No re-extraction needed — just point the dataset at the same files.

In [ ]:
from node_extractor_from_latent.dataset import NodeVocabulary, RepeatFactorSampler
from node_extractor_from_image.dataset_v3 import NodeDatasetPatched
from torch.utils.data import DataLoader

# Vocabulary (must exist from V1)
assert Path(cfg.vocab_path).exists(), (
    f"Vocabulary not found at {cfg.vocab_path}. "
    "Run train.ipynb (V1) first to build it."
)
vocab = NodeVocabulary()
vocab.load(cfg.vocab_path)

# Datasets — same patch cache as V3
assert Path(cfg.train_cache).exists(), (
    f"Patch cache not found at {cfg.train_cache}. "
    "Run train_v3.ipynb first to extract patch tokens."
)

train_ds = NodeDatasetPatched(cfg.train_jsonl, vocab, cfg.train_cache)
val_ds   = NodeDatasetPatched(cfg.val_jsonl,   vocab, cfg.val_cache)

rfs = RepeatFactorSampler(train_ds, vocab)

train_loader = DataLoader(
    train_ds, batch_size=cfg.batch_size,
    sampler=rfs,
    num_workers=cfg.num_workers, pin_memory=True,
    drop_last=True, persistent_workers=True,
)
val_loader = DataLoader(
    val_ds, batch_size=cfg.eval_batch_size, shuffle=False,
    num_workers=cfg.num_workers, pin_memory=True,
    persistent_workers=True,
)

patches, target = next(iter(train_loader))
print(f"Patch batch  : {patches.shape}   (B x 729 x 1152)")
print(f"Target shape : {target.shape}    (B x vocab)")
print(f"Avg pos/sample: {target.sum(dim=1).mean():.1f}")
print(f"Steps/epoch (RFS): {len(train_loader):,}")
print(f"Val batches  : {len(val_loader)}")

---
## 3. Initialize Model V4

In [ ]:
from node_extractor_from_image.model_v4 import NodeExtractorV4

model = NodeExtractorV4(
    patch_dim=cfg.patch_dim,
    vocab_size=vocab.size,
    hidden_dim=cfg.hidden_dim,
    num_blocks=cfg.num_blocks,
    expansion=cfg.expansion,
    dropout=cfg.dropout,
    input_dropout=cfg.input_dropout,
    pool_mode=cfg.pool_mode,
).to(device)

param_counts = model.count_parameters()
print("Parameter counts (V4 AttentionPool + Residual MLP):")
for name, count in param_counts.items():
    print(f"  {name:12s}: {count:>12,d}")

param_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**2
act_mb_est = (cfg.batch_size * cfg.hidden_dim * 4 * cfg.num_blocks * 3) / 1024**2
print(f"\nModel params      : ~{param_mb:.0f} MB")
print(f"Activation estimate: ~{act_mb_est:.0f} MB  (tiny — MLP after pooling)")

if cfg.compile_model and device == "cuda":
    try:
        model = torch.compile(model, mode="reduce-overhead")
        print("\ntorch.compile enabled")
    except Exception as e:
        print(f"\ntorch.compile failed: {e}")

if device == "cuda":
    print(f"\nGPU after model load: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

---
## 4. Loss, Optimizer & Scheduler

In [ ]:
from node_extractor_from_latent.utils import (
    AsymmetricLoss, compute_metrics, CosineWarmupScheduler, EarlyStopping,
)

criterion    = AsymmetricLoss(
    gamma_neg=cfg.gamma_neg, gamma_pos=cfg.gamma_pos,
    clip=cfg.clip, label_smoothing=cfg.label_smoothing,
)
optimizer    = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
total_steps  = len(train_loader) * cfg.epochs
warmup_steps = int(total_steps * cfg.warmup_ratio)
scheduler    = CosineWarmupScheduler(optimizer, warmup_steps, total_steps)
early_stopping = EarlyStopping(patience=cfg.patience, mode="max")

print(f"Total steps  : {total_steps:,}")
print(f"Warmup steps : {warmup_steps:,}")
print(f"Steps/epoch  : {len(train_loader):,}")

---
## 5. Training Loop

In [ ]:
scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)


def train_one_epoch(model, loader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss, n = 0.0, 0
    for patches, target in loader:
        patches = patches.to(device, non_blocking=True)
        target  = target.to(device,  non_blocking=True)
        with torch.amp.autocast("cuda", enabled=cfg.use_amp):
            loss = criterion(model(patches), target)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item(); n += 1
    return total_loss / n


@torch.no_grad()
def evaluate(model, loader, criterion, device, threshold=0.5):
    model.eval()
    total_loss, n = 0.0, 0
    all_logits, all_targets = [], []
    for patches, target in loader:
        patches = patches.to(device, non_blocking=True)
        target  = target.to(device,  non_blocking=True)
        with torch.amp.autocast("cuda", enabled=cfg.use_amp):
            logits = model(patches)
            loss   = criterion(logits, target)
        total_loss += loss.item(); n += 1
        all_logits.append(logits.float().cpu())
        all_targets.append(target.cpu())
    metrics = compute_metrics(torch.cat(all_logits), torch.cat(all_targets), threshold)
    metrics["loss"] = total_loss / n
    return metrics


print("Train/eval functions ready.")

In [ ]:
history = {k: [] for k in [
    "train_loss", "val_loss", "mAP", "macro_mAP",
    "micro_f1", "macro_f1", "micro_precision", "micro_recall",
    "lr", "gpu_mem_gb", "epoch_time",
]}
best_map = 0.0
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
plt.ion()

for epoch in range(1, cfg.epochs + 1):
    t0 = time.time()
    rfs.set_epoch(epoch)

    train_loss  = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, device)
    val_metrics = evaluate(model, val_loader, criterion, device, cfg.eval_threshold)

    elapsed = time.time() - t0
    gpu_mem = torch.cuda.max_memory_allocated() / 1024**3 if device == "cuda" else 0
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()

    current_lr = optimizer.param_groups[0]["lr"]
    for k, v in [("train_loss", train_loss), ("val_loss", val_metrics["loss"]),
                 ("mAP", val_metrics["mAP"]), ("macro_mAP", val_metrics["macro_mAP"]),
                 ("micro_f1", val_metrics["micro_f1"]), ("macro_f1", val_metrics["macro_f1"]),
                 ("micro_precision", val_metrics["micro_precision"]),
                 ("micro_recall", val_metrics["micro_recall"]),
                 ("lr", current_lr), ("gpu_mem_gb", gpu_mem), ("epoch_time", elapsed)]:
        history[k].append(v)

    if val_metrics["mAP"] > best_map:
        best_map = val_metrics["mAP"]
        torch.save({
            "epoch": epoch, "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "mAP": best_map, "config": cfg, "version": "v4",
        }, best_path)

    clear_output(wait=True)
    for ax in axes: ax.clear()
    ex = range(1, epoch + 1)

    axes[0].plot(ex, history["train_loss"], "b-", label="Train", linewidth=1.5)
    axes[0].plot(ex, history["val_loss"],   "r-", label="Val",   linewidth=1.5)
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(ex, history["mAP"],       "g-",  label="Micro mAP", linewidth=1.5)
    axes[1].plot(ex, history["macro_mAP"], "g--", label="Macro mAP", linewidth=1.5, alpha=0.6)
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("mAP")
    axes[1].set_title(f"Mean Average Precision  (best={best_map:.4f})")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    axes[2].plot(ex, history["micro_f1"],       "m-",  label="Micro F1",  linewidth=1.5)
    axes[2].plot(ex, history["micro_precision"], "c--", label="Precision", linewidth=1.2)
    axes[2].plot(ex, history["micro_recall"],    "y--", label="Recall",    linewidth=1.2)
    axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Score")
    axes[2].set_title("F1 / Precision / Recall")
    axes[2].legend(); axes[2].grid(True, alpha=0.3)

    is_best = " *" if val_metrics["mAP"] >= best_map else ""
    fig.suptitle(
        f"[V4 AttnPool+MLP]  Epoch {epoch}/{cfg.epochs}  |  "
        f"mAP={val_metrics['mAP']:.4f}{is_best}  |  "
        f"F1={val_metrics['micro_f1']:.4f}  |  "
        f"{len(rfs)/elapsed:.0f} samp/s  |  "
        f"GPU={gpu_mem:.1f}GB  |  lr={current_lr:.1e}",
        fontsize=11, fontweight="bold",
    )
    plt.tight_layout(); display(fig)

    if early_stopping(val_metrics["mAP"], epoch):
        print(f"\nEarly stopping at epoch {epoch}. "
              f"Best mAP={best_map:.4f} at epoch {early_stopping.best_epoch}")
        break

plt.ioff()
avg_time = np.mean(history["epoch_time"])
print(f"\n{'='*60}")
print(f"  V4 Training complete")
print(f"{'='*60}")
print(f"  Best mAP       : {best_map:.4f}")
print(f"  Avg epoch time : {avg_time:.1f}s")
print(f"  Peak GPU mem   : {max(history['gpu_mem_gb']):.1f} GB")
print(f"  Checkpoint     : {best_path}")

---
## 6. Evaluation — Per-class AP Analysis

In [ ]:
from sklearn.metrics import average_precision_score

checkpoint = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded best_model_v4.pt — epoch {checkpoint['epoch']}, mAP={checkpoint['mAP']:.4f}")

model.eval()
all_logits, all_targets = [], []
with torch.no_grad():
    for patches, target in val_loader:
        all_logits.append(model(patches.to(device)).cpu())
        all_targets.append(target)

all_logits  = torch.cat(all_logits)
all_targets = torch.cat(all_targets)
all_probs   = torch.sigmoid(all_logits).numpy()
all_tgts_np = all_targets.numpy()

has_pos = all_tgts_np.sum(axis=0) > 0
per_class_ap = np.zeros(vocab.size)
for i in range(vocab.size):
    if has_pos[i]:
        per_class_ap[i] = average_precision_score(all_tgts_np[:, i], all_probs[:, i])

sorted_idx = np.argsort(per_class_ap)[::-1]

print(f"\n{'='*65}")
print(f"  TOP 20  (by AP)")
print(f"{'='*65}")
print(f"  {'Rank':>4}  {'Label':>22}  {'AP':>8}  {'Support':>8}")
for rank, idx in enumerate(sorted_idx[:20]):
    print(f"  {rank+1:4d}  {vocab.idx_to_label[idx]:>22s}  "
          f"{per_class_ap[idx]:8.4f}  {int(all_tgts_np[:, idx].sum()):8d}")

print(f"\n{'='*65}")
print(f"  BOTTOM 20  (support > 0)")
print(f"{'='*65}")
for rank, idx in enumerate([i for i in sorted_idx[::-1] if has_pos[i]][:20]):
    print(f"  {rank+1:4d}  {vocab.idx_to_label[idx]:>22s}  "
          f"{per_class_ap[idx]:8.4f}  {int(all_tgts_np[:, idx].sum()):8d}")

valid_aps = per_class_ap[has_pos]
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(valid_aps, bins=50, color="steelblue", edgecolor="white")
ax.axvline(valid_aps.mean(), color="red", linestyle="--",
           label=f"Mean AP = {valid_aps.mean():.3f}")
ax.set_xlabel("Average Precision")
ax.set_ylabel("Number of classes")
ax.set_title("V4 — Per-Class AP Distribution (AttentionPool + MLP)")
ax.legend(); plt.tight_layout(); plt.show()

---
## 7. Inference Demo — Prediction Visualization

In [ ]:
import json
from PIL import Image
from matplotlib.patches import Patch
from node_extractor_from_latent.utils import normalize_label

with open(cfg.val_jsonl) as f:
    val_lines = f.readlines()

NUM_VIS = 4
random.seed(42)
vis_indices = random.sample(range(len(val_lines)), NUM_VIS)

model.eval()

for sample_idx in vis_indices:
    entry          = json.loads(val_lines[sample_idx])
    gt_labels_raw  = [node["label"] for node in entry["groundTruth"]["nodes"]]
    gt_labels_norm = set(normalize_label(l) for l in gt_labels_raw)

    patches_emb, _ = val_ds[sample_idx]
    with torch.no_grad():
        probs = torch.sigmoid(
            model(patches_emb.unsqueeze(0).to(device))
        ).squeeze(0).cpu()

    top_k = 25
    top_scores, top_idx = torch.topk(probs, top_k)
    pred_labels = [vocab.idx_to_label[i.item()] for i in top_idx]
    pred_scores = top_scores.tolist()
    pred_hits   = [label in gt_labels_norm for label in pred_labels]

    all_preds = vocab.decode(probs, threshold=cfg.eval_threshold)
    pred_set  = set(l for l, _ in all_preds)
    tp        = len(pred_set & gt_labels_norm)
    precision = tp / len(pred_set)       if pred_set        else 0
    recall    = tp / len(gt_labels_norm) if gt_labels_norm  else 0
    f1        = 2*precision*recall / (precision+recall) if (precision+recall) > 0 else 0

    fig = plt.figure(figsize=(18, 7))
    gs  = fig.add_gridspec(1, 2, width_ratios=[1, 1.4], wspace=0.05)

    ax_img = fig.add_subplot(gs[0])
    if entry.get("image_path") and Path(entry["image_path"]).exists():
        ax_img.imshow(Image.open(entry["image_path"]))
    else:
        ax_img.text(0.5, 0.5, "Image not available", ha="center", va="center",
                    fontsize=14, transform=ax_img.transAxes)
    ax_img.set_title(
        f"{entry['id']}  ({entry['metadata']['source'].upper()})\n"
        f"GT: {len(gt_labels_norm)} entities  |  Pred: {len(all_preds)} above {cfg.eval_threshold}",
        fontsize=11, fontweight="bold", pad=10,
    )
    ax_img.axis("off")
    gt_text = ", ".join(sorted(gt_labels_norm)[:20])
    if len(gt_labels_norm) > 20:
        gt_text += f"  (+{len(gt_labels_norm)-20} more)"
    ax_img.text(
        0.5, -0.02, f"Ground Truth: {gt_text}",
        ha="center", va="top", fontsize=8, color="#444",
        transform=ax_img.transAxes, wrap=True,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#f0f0f0", edgecolor="#ccc"),
    )

    ax_bar = fig.add_subplot(gs[1])
    y_pos  = np.arange(top_k)[::-1]
    colors = ["#2ecc71" if h else "#e74c3c" for h in pred_hits]
    ax_bar.barh(y_pos, pred_scores, color=colors,
                edgecolor=["#27ae60" if h else "#c0392b" for h in pred_hits], height=0.7)
    ax_bar.axvline(cfg.eval_threshold, color="#555", linestyle="--", linewidth=1, alpha=0.7)
    ax_bar.text(cfg.eval_threshold+0.01, top_k-0.5,
                f"threshold={cfg.eval_threshold}", fontsize=8, color="#555", va="bottom")

    for i, (label, score, hit) in enumerate(zip(pred_labels, pred_scores, pred_hits)):
        marker = "\u2713" if hit else "\u2717"
        ax_bar.text(0.01, y_pos[i], f"  {marker} {label}", va="center", ha="left",
                    fontsize=9, fontweight="bold" if hit else "normal",
                    color="white" if score > 0.3 else "#333")
        ax_bar.text(score+0.008, y_pos[i], f"{score:.3f}",
                    va="center", ha="left", fontsize=8, color="#333")

    ax_bar.set_xlim(0, 1.05); ax_bar.set_yticks([])
    ax_bar.set_xlabel("Confidence Score", fontsize=10)
    ax_bar.set_title(
        f"Top {top_k} Predictions  [V4 AttnPool+MLP]  |  "
        f"P={precision:.2f}  R={recall:.2f}  F1={f1:.2f}",
        fontsize=11, fontweight="bold", pad=10,
    )
    for spine in ["top", "right", "left"]:
        ax_bar.spines[spine].set_visible(False)
    ax_bar.legend(
        handles=[Patch(facecolor="#2ecc71", label="In ground truth"),
                 Patch(facecolor="#e74c3c", label="Not in ground truth")],
        loc="lower right", fontsize=9, framealpha=0.9,
    )
    plt.tight_layout(); plt.show(); print()